In [4]:
!pip install mysql-connector-python

Connect to the database

In [2]:
import mysql.connector
import getpass
db_user = getpass.getpass('Enter username')
db_password = getpass.getpass('Enter password')
config = {
   'user': db_user,
   'password': db_password,
   'host': 'localhost',
   'port': 3306,
   'database': '461proj',
   'raise_on_warnings': True                    
}
db_connection = mysql.connector.connect(**config)

Enter username ········
Enter password ········


Update the attributes of a property

In [22]:
cur = db_connection.cursor()
property_number = int(input("Enter property number to update: "))
new_price = int(input("Enter monthly price: "))
new_fee = int(input("Enter management fee: "))

new_adv = input("Is property advertised? (True/False): ")
new_adv = True if new_adv.lower() == "true" else False

query = """
UPDATE rental_properties
SET monthly_price = %s, adv = %s, management_fee = %s
WHERE property_number = %s
"""
values = (new_price, new_adv, new_fee, property_number)
cur.execute(query, values)
db_connection.commit()
cur.close()
print("Property updated successfully")

Enter property number to update:  1
Enter monthly price:  1
Enter management fee:  1
Is property advertised? (True/False):  true


True

Create a new client

In [29]:
cur = db_connection.cursor()
client_number = int(input("Enter client number: "))
first_name = str(input("Enter first name: "))
last_name = str(input("Enter last name: "))
street = str(input("Enter street: "))
city = str(input("Enter city: "))
state = str(input("Enter state: "))
zipcode = int(input("Enter zipcode: "))
unit_number = int(input("Enter unit number: "))
email = str(input("Enter email: "))
max_rent = int(input("Enter max rent: "))
preference = str(input("Enter preference: "))

query = """
INSERT INTO clients VALUES
(%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""
values = (client_number, first_name, last_name, street, city, state, zipcode, unit_number, email, max_rent, preference)
cur.execute(query, values)
db_connection.commit()
cur.close()
print("Client added successfully")

Enter client number:  13
Enter first name:  a
Enter last name:  b
Enter street:  c
Enter city:  d
Enter state:  e
Enter zipcode:  123
Enter unit number:  34
Enter email:  g
Enter max rent:  34
Enter preference:  e


True

Create a new property viewing

In [34]:
cur = db_connection.cursor()
employee_number = int(input("Enter employee number: "))
client_number = int(input("Enter client number: "))
property_number = int(input("Enter property number: "))
view_time = str(input("Enter view time: "))
view_date = str(input("Enter view date: "))


query = """
INSERT INTO property_viewings VALUES
(%s, %s, %s, %s, %s);
"""
values = (employee_number, client_number, property_number, view_time, view_date)
cur.execute(query, values)
db_connection.commit()
cur.close()
print("Property viewing added successfully")

Enter employee number:  3
Enter client number:  2
Enter property number:  4
Enter view time:  10:00
Enter view date:  2023-2-12


True

Discard a property

In [37]:
cur = db_connection.cursor()
property_number = int(input("Enter property number to remove: "))

query = """
DELETE FROM property_viewings WHERE property_number = %s
"""
cur.execute(query, (property_number,))

query = """
DELETE FROM assigned WHERE property_number = %s
"""
cur.execute(query, (property_number,))

query = """
DELETE FROM involve WHERE property_number = %s
"""
cur.execute(query, (property_number,))

query = """
DELETE FROM owned_by WHERE property_number = %s
"""
cur.execute(query, (property_number,))

query = """
DELETE FROM rent_out WHERE property_number = %s
"""
cur.execute(query, (property_number,))

query = """
DELETE FROM rental_properties WHERE property_number = %s
"""
cur.execute(query, (property_number,))

db_connection.commit()
cur.close()
print("Property removed successfully")

Enter property number to remove:  2


Property removed successfully


Change the partner assigned to an owner and transfer all current (non-expired)
leases of that owner to the new partner.

In [13]:
cur = db_connection.cursor()
owner_number = int(input("Enter owner number: "))
partner_number = int(input("Enter new partner employee number: "))

query1 = """
UPDATE assigned_to
SET employee_number = %s
WHERE owner_number = %s
"""

query2 = """
UPDATE leases
JOIN rent_out ON leases.lease_number = rent_out.lease_number
JOIN owned_by ON rent_out.property_number = owned_by.property_number
SET leases.employee_number = %s
WHERE owned_by.owner_number = %s
AND leases.end_date >= CURDATE()
"""
values = (partner_number, owner_number)

try:
    cur.execute("SET TRANSACTION ISOLATION LEVEL SERIALIZABLE")
    cur.execute("START TRANSACTION;")
    cur.execute(query1, values)
    cur.execute(query2, values)
    cur.execute("COMMIT")
    print("New partner updated successfully")
except Exception as e:
    cur.execute("ROLLBACK")
    print("rolled back")
finally:
    cur.close()


Enter owner number:  1
Enter new partner employee number:  3


New partner updated successfully


Disconnect from the database

In [24]:
db_connection.close()